# Single-stage receive coil amplifier

In [1]:
import SLiCAP as sl
from sympy import pi, sqrt, log, Abs
sl.initProject('Hearing loop', notebook=True)
specs     = sl.csv2specs("fb_concept.csv")
spec_dict = sl.specs2dict(specs)
# import operating point information from the last run of 'firstStageDesign.ipynb'.
%store -r op_info 
%store -r channel
channel = channel.upper()
V_GS    = Abs(op_info["V_G"])
g_m     = op_info["g_m"]
c_iss   = op_info["c_iss"]

Compiling library: SLiCAP.lib.


Compiling library: SLiCAPmodels.lib.


## Single-stage motivation

A single-stage solution is possible if the noise design parameters of the input stage and the drive design parameters of the output stage overlap.

We can check this by setting the current of the input stage to the peak current drive requirement of the output stage.

The peak current drive requitement $I_{p}$ is

\begin{equation}
I_{p}=\frac{Vi_{pp}}{2 R_f}
\end{equation}

The controller of the receive coil amplifier needs to be a four-terminal type. Hence, a differential pair
or its cascoded version, are the only valid candidates for a single-stage amplifier.

The peak-peak output voltage $Vo_{pp}$ maximally amounts

\begin{equation}
Vo_{pp}=Vi_{pp} \sqrt{1+\frac{1}{4 \pi^2 f_{fp}^2 \tau_i^2}}
\end{equation}

In [2]:
I_p = spec_dict["Vi_pp"].value/(2*spec_dict["R_f"].value)
sl.eqn2html("I_p", I_p, units="A")

In [3]:
Vo_pp = spec_dict["Vi_pp"].value*sqrt(1 + 1/(2*pi*spec_dict["f_fp"].value * spec_dict["tau_i"].value)**2 )
sl.eqn2html("Vo_pp", Vo_pp, units="V")

This fits well onto the ADC input voltage range and leaves a total headroom $V_{sat_{tot}}$ for biasing of
\begin{equation}
V_{sat_{tot}} = V_{ADC_{high}}-V_{ADC_{low}}-Vo_{pp}
\end{equation}

In [4]:
V_sat_tot = spec_dict["V_ADC_high"].value - spec_dict["V_ADC_low"].value - Vo_pp
sl.eqn2html("V_sat_tot", V_sat_tot)

## Stage design
### Current drive check of noise optimized input stage
The operating current for a PMOS CS stage input stage was $I_{iP2}$. The total operating current of a differential pair with the same noise performance must be $4\times I_{iP2}$, and the peak output current $I_{op}$ of such a stage can be $2\times I_{iP2}$:

In [5]:
I_op = Abs(2 * spec_dict["I_i{}2".format(channel.upper())].value)

if I_op > I_p:
    print("The current drive capability of a {}MOS differentail-pair is OK!".format(channel.upper()))
else:
    print("The current drive capapility is {:.3g}, and must be increased to {:.3g}".format(float(I_op), float(I_p)))

The current drive capapility is 8.07e-06, and must be increased to 1.65e-05


### Voltage drive check of noise optimized input stage
The saturation voltage of the input stage transistor can be estimated from the operating point information 
and the threshold voltage of the input-stage transistor (about 0.43V for N and P channel devices). Since we also need to bias the differential pair, we need more than this saturation voltage! We will check for two times this saturation voltage.

In [6]:
sl.eqn2html("V_GS{}".format(channel.upper()), V_GS)

In [7]:
V_th = 0.43
if 2 * (V_GS - V_th) <= V_sat_tot/2:
    print("The voltage drive capability of a {}MOS differentail-pair is OK!".format(channel.upper()))
else:
    print("The votage drive capapility is {:.3g}, and must be increased to {:.3g}".format(float(2 * (V_GS - V_th)), float(V_sat_tot/2)))

The voltage drive capability of a PMOS differentail-pair is OK!


## Bandwidth and accuracy check of a single-stage amplifier

In [8]:
cir = sl.makeCircuit("kicad/A1/singleStageSimple.kicad_sch")
sl.img2html("singleStageSimple.svg", 600)

Creating netlist of kicad/A1/singleStageSimple.kicad_sch using KiCAD


Creating drawing-size SVG and PDF images of kicad/A1/singleStageSimple.kicad_sch


Plotted to '/home/danieltyukov/workspace/tud/tud-structured-electronic-design/Notebooks/img/singleStageSimple.svg'.
Done.


Checking netlist: cir/singleStageSimple.cir


### Define the circuit parameters
To obtain equal noise performance as the CS stage, the small-signal parameters of the differential pair must equal those of the CS stage. This is at the costs of four times current and area.

In [9]:
sl.specs2circuit(specs, cir)
cir.defPars(op_info)
# Check for numeric values
sl.elementData2html(cir)

RefDes,Nodes,Refs,Model,Param,Symbolic,Numeric
C1,2 1,,C,value,$c_{iss}$,$13.96\cdot 10^{-15}$
,,,,vinit,$0$,$0$
C2,out 0,,C,value,$C_{L}$,$10\cdot 10^{-12}$
,,,,vinit,$0$,$0$
C3,out 1,,C,value,$C_{i}$,$4.955\cdot 10^{-9}$
,,,,vinit,$0$,$0$
C4,2 0,,C,value,$C_{s}$,$9.382\cdot 10^{-12}$
,,,,vinit,$0$,$0$
G1,out 0 2 1,,G,value,$- g_{m}$,$-73.46\cdot 10^{-6}$
L1,2 3,,L,value,$L_{s}$,$0.12$


## Loop gain analysis
### Pole-zero analysis
Pole-zero analysis of the loop gain yields the DC loop gain and its poles and zeros. From this data, we can analyse
the accuracy and the bandwidth of the amplifier.

In [10]:
PZresult = sl.doPZ(cir, pardefs="circuit", transfer="loopgain")
sl.pz2html(PZresult)

pole,Re [Hz],Im [Hz],Mag [Hz],Q
p1,$-598.8$,,$598.8$,
p2,$-106.5\cdot 10^{3}$,$106.6\cdot 10^{3}$,$150.7\cdot 10^{3}$,$0.7076$
p3,$-106.5\cdot 10^{3}$,$-106.6\cdot 10^{3}$,$150.7\cdot 10^{3}$,$0.7076$
p4,$-4.913\cdot 10^{6}$,,$4.913\cdot 10^{6}$,
zero,Re [Hz],Im [Hz],Mag [Hz],Q
z1,$-600$,,$600$,
z2,$-106.6\cdot 10^{3}$,$106.6\cdot 10^{3}$,$150.8\cdot 10^{3}$,$0.7071$
z3,$-106.6\cdot 10^{3}$,$-106.6\cdot 10^{3}$,$150.8\cdot 10^{3}$,$0.7071$


### Symbolic expression of DC loopgain

In [11]:
L_DC = sl.doLaplace(cir, transfer="loopgain").laplace.subs(sl.ini.laplace, 0)
sl.eqn2html("L_DC", L_DC)

### Plots asymptotic-gain model

In [12]:
AG = sl.doLaplace(cir, transfer="asymptotic", pardefs="circuit")
LG = sl.doLaplace(cir, transfer="loopgain",   pardefs="circuit")
SG = sl.doLaplace(cir, transfer="servo",      pardefs="circuit")
DG = sl.doLaplace(cir, transfer="direct",     pardefs="circuit")
GG = sl.doLaplace(cir, pardefs="circuit")
transfers = [AG, LG, SG, DG, GG]
sl.plotSweep("feedbackMag1", "Magnitude plots", transfers, 0.1, 10e3, 200, sweepScale="k")
sl.img2html("feedbackMag1.svg", 600)

## Questions

1. Please explain the DC value of the loop gain and the frequencies of the poles and the zeros?
2. What happens with the loop gain if we use a more elaborate model of the differential pair that includes $g_o$ and $c_{dg}$?
3. How could we improve the accuracy of the transfer?
4. Would you like to continue with the design of a single-stage solution? Please motivate your answer!
5. Please formulate your next design step.